In [3]:
!pip install ultralytics

In [17]:
!mkdir -p /content/data/kitti

In [21]:
!wget -P /content/data/kitti https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_image_2.zip

--2026-07-16 09:32:28--  https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_image_2.zip
Resolving s3.eu-central-1.amazonaws.com (s3.eu-central-1.amazonaws.com)... 16.12.32.5, 3.5.120.188, 3.5.122.102, ...
Connecting to s3.eu-central-1.amazonaws.com (s3.eu-central-1.amazonaws.com)|16.12.32.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12569945557 (12G) [application/zip]
Saving to: ‘/content/data/kitti/data_object_image_2.zip.1’

data_object_image_2 100%[===================>]  11.71G  23.2MB/s    in 8m 53s  

2026-07-16 09:41:22 (22.5 MB/s) - ‘/content/data/kitti/data_object_image_2.zip.1’ saved [12569945557/12569945557]



In [19]:
!wget -P /content/data/kitti https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_label_2.zip

--2026-07-16 09:30:08--  https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_label_2.zip
Resolving s3.eu-central-1.amazonaws.com (s3.eu-central-1.amazonaws.com)... 3.5.122.199, 3.5.135.169, 3.5.120.42, ...
Connecting to s3.eu-central-1.amazonaws.com (s3.eu-central-1.amazonaws.com)|3.5.122.199|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5601213 (5.3M) [application/zip]
Saving to: ‘/content/data/kitti/data_object_label_2.zip’

data_object_label_2 100%[===================>]   5.34M  4.62MB/s    in 1.2s    

2026-07-16 09:30:09 (4.62 MB/s) - ‘/content/data/kitti/data_object_label_2.zip’ saved [5601213/5601213]



In [20]:
!unzip -q /content/data/kitti/data_object_image_2.zip -d /content/data/kitti
!unzip -q /content/data/kitti/data_object_label_2.zip -d /content/data/kitti

In [23]:
from pathlib import Path

base = Path("/content/data/kitti")

for p in [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]:
    (base/p).mkdir(parents=True, exist_ok=True)

In [24]:
from pathlib import Path
from PIL import Image
import shutil
import random

src = Path("/content/data/kitti/training")
out = Path("/content/data/kitti")

classes = {
    "Car":0,
    "Van":1,
    "Truck":2,
    "Pedestrian":3,
    "Person_sitting":4,
    "Cyclist":5,
    "Tram":6,
    "Misc":7
}

images = list((src/"image_2").glob("*.png"))

random.seed(42)
random.shuffle(images)

split = int(len(images)*0.8)

sets = {
    "train": images[:split],
    "val": images[split:]
}

for name, imgs in sets.items():

    for img in imgs:

        label = src/"label_2"/(img.stem + ".txt")

        shutil.copy(
            img,
            out/f"images/{name}"/img.name
        )

        yolo_file = out/f"labels/{name}"/f"{img.stem}.txt"

        with Image.open(img) as im:
            w,h = im.size

        lines=[]

        if label.exists():
            for row in label.read_text().splitlines():

                data=row.split()

                cls=data[0]

                if cls not in classes:
                    continue

                xmin=float(data[4])
                ymin=float(data[5])
                xmax=float(data[6])
                ymax=float(data[7])

                xc=((xmin+xmax)/2)/w
                yc=((ymin+ymax)/2)/h
                bw=(xmax-xmin)/w
                bh=(ymax-ymin)/h

                lines.append(
                    f"{classes[cls]} {xc} {yc} {bw} {bh}"
                )

        yolo_file.write_text("\n".join(lines))

In [27]:
from pathlib import Path

yaml_content = """
path: /content/data/kitti

train: images/train
val: images/val
test: images/test

names:
  0: Car
  1: Van
  2: Truck
  3: Pedestrian
  4: Person_sitting
  5: Cyclist
  6: Tram
  7: Misc
"""

Path("configs/kitti.yaml").write_text(yaml_content)

print(Path("configs/kitti.yaml").read_text())


path: /content/data/kitti

train: images/train
val: images/val
test: images/test

names:
  0: Car
  1: Van
  2: Truck
  3: Pedestrian
  4: Person_sitting
  5: Cyclist
  6: Tram
  7: Misc



In [28]:
# 1/10 - Setup and smoke train to epoch 1
from pathlib import Path
import shutil
from ultralytics import YOLO

DATA_CONFIG = "configs/kitti.yaml"
MODEL_WEIGHTS = "yolo26n.pt"
IMAGE_SIZE = 640
DEVICE = "0"
WEIGHTS_DIR = Path("weights")
TENSORBOARD_DIR = Path("tensorboard")
WEIGHTS_DIR.mkdir(exist_ok=True)
TENSORBOARD_DIR.mkdir(exist_ok=True)

model = YOLO(MODEL_WEIGHTS)
results_1 = model.train(data=DATA_CONFIG, epochs=1, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_001", exist_ok=True)
current_weights = Path(results_1.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_001.pt")
print("Saved:", WEIGHTS_DIR / "speedereye_epoch_001.pt")
print("TensorBoard: %load_ext tensorboard ; %tensorboard --logdir tensorboard")


Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=epoch_001, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [29]:
# 2/10 - Continue to epoch 10
model = YOLO(str(current_weights))
results_2 = model.train(data=DATA_CONFIG, epochs=9, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_010", exist_ok=True)
current_weights = Path(results_2.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_010.pt")


Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=9, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/tensorboard/epoch_001/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=epoch_010, nbs=64, nms=False, opset=None, optimiz

PosixPath('weights/speedereye_epoch_010.pt')

In [ ]:
# 3/10 - Continue to epoch 20
model = YOLO(str(current_weights))
results_3 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_020", exist_ok=True)
current_weights = Path(results_3.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_020.pt")


Ultralytics 8.4.96 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=configs/kitti.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/detect/tensorboard/epoch_010/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=epoch_020, nbs=64, nms=False, opset=None, optimi

In [ ]:
# 4/10 - Continue to epoch 30
model = YOLO(str(current_weights))
results_4 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_030", exist_ok=True)
current_weights = Path(results_4.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_030.pt")


In [ ]:
# 5/10 - Continue to epoch 40
model = YOLO(str(current_weights))
results_5 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_040", exist_ok=True)
current_weights = Path(results_5.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_040.pt")


In [ ]:
# 6/10 - Continue to epoch 50
model = YOLO(str(current_weights))
results_6 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_050", exist_ok=True)
current_weights = Path(results_6.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_050.pt")


In [ ]:
# 7/10 - Continue to epoch 60
model = YOLO(str(current_weights))
results_7 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_060", exist_ok=True)
current_weights = Path(results_7.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_060.pt")


In [ ]:
# 8/10 - Continue to epoch 70
model = YOLO(str(current_weights))
results_8 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_070", exist_ok=True)
current_weights = Path(results_8.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_070.pt")


In [ ]:
# 9/10 - Continue to epoch 80
model = YOLO(str(current_weights))
results_9 = model.train(data=DATA_CONFIG, epochs=10, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_080", exist_ok=True)
current_weights = Path(results_9.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_080.pt")


In [ ]:
# 10/10 - Continue to epoch 100 and open TensorBoard
model = YOLO(str(current_weights))
results_10 = model.train(data=DATA_CONFIG, epochs=20, imgsz=IMAGE_SIZE, device=DEVICE, project=str(TENSORBOARD_DIR), name="epoch_100", exist_ok=True)
current_weights = Path(results_10.save_dir) / "weights" / "last.pt"
shutil.copy2(current_weights, WEIGHTS_DIR / "speedereye_epoch_100.pt")

%load_ext tensorboard
%tensorboard --logdir tensorboard
